In [11]:
from pathlib import Path

from cellgrn.main import (
    compute_all_cells_grn,
    format_celltype_grn,
    format_sample_grn,
    normalize_rna,
    parse_edges,
    summarize_grn,
)

import os
import pickle

import anndata as ad
import numpy as np
import pandas as pd
from scipy import sparse

#### **Step-by-step CellGRN example on the melanoma dataset**

This notebook is the second example in the repository. The root-level `tutorial.ipynb` is a compact CellGRN and CellGRN-sparse demonstration on a 500-cell multiome dataset. This notebook focuses on the melanoma cell line dataset used in the manuscript and runs CellGRN with both LINGER and SCENIC+ backbones.

1. First, preprocess the output of a GRN inference method, such as SCENIC+, with `code/preprocess/prepare_melanoma_input.ipynb` to define the input edges, genes, TFs, and peaks.
2. Next, load RNA and ATAC h5ad files and the processed GRN backbone files. The metadata table should include cell type information.
3. Finally, CellGRN outputs cell-specific, sample-wise, and cell-type-wise GRN results. This example requires CPU only.

**Note:**  
The input raw and processed dataset is available at Figshare: https://doi.org/10.6084/m9.figshare.31304758. You can use the processed data as direct input for this demo. After uncompressing the data, update `MELANOMA_DIR` below to match your local directory layout.

In [ ]:
# Please change this input path on your laptop/server.
MELANOMA_DIR = Path("/path/to/uncompressed/cellGRN/data/melanoma")

# Resolve the repository root whether the notebook is launched from the repo root or from code/.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "all_hg_TF.txt").exists() and (PROJECT_ROOT.parent / "all_hg_TF.txt").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

OUTPUT_ROOT = PROJECT_ROOT / "output"
TF_FILE = PROJECT_ROOT / "all_hg_TF.txt"

cell_meta = pd.read_csv(MELANOMA_DIR / "metadata.csv", index_col=0)

input_rna = ad.read_h5ad(MELANOMA_DIR / "Melanoma-cell_line-RNA-counts.h5ad")
input_atac = ad.read_h5ad(MELANOMA_DIR / "Melanoma-cell_line-ATAC-peaks.h5ad")

all_tf = [i.rstrip() for i in open(TF_FILE)]

In [4]:
input_gene = input_rna.var.index.values
input_peak = input_atac.var.index.values
input_tf = list(set(input_gene) & set(all_tf))

# Align metadata to the h5ad cell order before summarizing by cell type.
cell_meta = cell_meta.loc[input_rna.obs_names].copy()
cell_types = cell_meta["cell_type"]

In [ ]:
for soft in ["linger", "scenic2"]:
    rna_counts = input_rna.X.toarray() if sparse.issparse(input_rna.X) else input_rna.X
    atac_counts = input_atac.X.toarray() if sparse.issparse(input_atac.X) else input_atac.X

    input_df1 = pd.DataFrame(rna_counts, index=input_rna.obs.index.values, columns=input_rna.var.index.values)
    peak_rename = [i.replace("-", ":", 1) for i in input_atac.var.index.values]
    input_df2 = pd.DataFrame(atac_counts, index=input_atac.obs.index.values, columns=peak_rename)

    outdir = OUTPUT_ROOT / f"res_melanoma_{soft}"
    outdir.mkdir(parents=True, exist_ok=True)

    cand_df = pd.read_csv(MELANOMA_DIR / f"{soft}_grn.csv", header=0)

    input_genes = [i.rstrip() for i in open(MELANOMA_DIR / f"{soft}_gene.txt")]
    input_peaks = [i.rstrip() for i in open(MELANOMA_DIR / f"{soft}_peak.txt")]

    input_df1 = input_df1[input_genes]
    input_df2 = input_df2[input_peaks]

    rna_data1, rna_data2 = normalize_rna(input_df1)
    atac_data = input_df2.copy()

    input_tfs = [tf for tf in input_tf if tf in input_genes]
    tf_data1 = rna_data1[input_tfs].copy()
    tf_data2 = rna_data2[input_tfs].copy()

    edges_idx, edges_name = parse_edges(cand_df, input_tfs, input_genes, input_peaks)

    grn_scale2 = compute_all_cells_grn(
        tf_data2,
        rna_data2,
        atac_data,
        edges_idx,
        edges_name,
        input_tfs,
        input_genes,
        input_peaks,
    )

    with open(outdir / f"{soft}_cell_grn.pkl", "wb") as f:
        pickle.dump(grn_scale2.copy(), f)

    sample_grn_scale2, celltype_grn_scale2 = summarize_grn(grn_scale2, cell_types)

    tf_gene_res_scale2, tf_peak_res_scale2, gene_peak_res_scale2 = format_sample_grn(sample_grn_scale2)
    tf_gene_ct_res_scale2, tf_peak_ct_res_scale2, gene_peak_ct_res_scale2 = format_celltype_grn(celltype_grn_scale2)

    tf_gene_res_scale2.to_csv(outdir / "tf_gene_sample_scale2.csv", index=False)
    tf_peak_res_scale2.to_csv(outdir / "tf_peak_sample_scale2.csv", index=False)
    gene_peak_res_scale2.to_csv(outdir / "gene_peak_sample_scale2.csv", index=False)

    tf_gene_ct_res_scale2.to_csv(outdir / "tf_gene_celltype_scale2.csv", index=False)
    tf_peak_ct_res_scale2.to_csv(outdir / "tf_peak_celltype_scale2.csv", index=False)
    gene_peak_ct_res_scale2.to_csv(outdir / "gene_peak_celltype_scale2.csv", index=False)